### 1. Imports and Configuration

In [ ]:
import math
import time

import numpy               as np
import torch
import torch.nn            as nn
import torch.nn.functional as F

torch.manual_seed(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# -----------------------------------------------------------------------
# Configuration.
# -----------------------------------------------------------------------

DIM          = 768
NUM_HEADS    = 8
NUM_KV_HEADS = 4
HEAD_DIM     = DIM // NUM_HEADS
MLP_MULT     = 2
HIDDEN       = DIM * MLP_MULT
VOCAB        = 8192
EMBED_DIM    = 256
SEQ_LEN      = 128
BATCH        = 32
N_LAYERS     = 3
N_STEPS      = 400
LR           = 3e-4
MAX_BYTES    = 10

print(f'Device  : {device}')
print(f'Vocab   : {VOCAB}  |  Max bytes/token: {MAX_BYTES}')

### 2. BPE Token Byte Decomposition

In [ ]:
# -----------------------------------------------------------------------
# Simulate sp8192 token -> byte decomposition.
# Token 0-255: single-byte identities.
# Token 256+:  2 to MAX_BYTES bytes, pseudo-random seeded by token ID.
# -----------------------------------------------------------------------

np.random.seed(42)
token_byte_seqs = []
for t in range(VOCAB):
    if t < 256:
        token_byte_seqs.append([t])
    else:
        rng     = np.random.RandomState(t)
        n_bytes = min(MAX_BYTES, 2 + (t - 256) // 1500)
        token_byte_seqs.append(rng.randint(0, 256, size=n_bytes).tolist())

byte_matrix  = torch.zeros(VOCAB, MAX_BYTES, dtype=torch.long)
byte_lengths = torch.zeros(VOCAB, dtype=torch.long)
for t in range(VOCAB):
    bts              = token_byte_seqs[t]
    byte_lengths[t]  = len(bts)
    for i, b in enumerate(bts):
        byte_matrix[t, i] = b

byte_matrix  = byte_matrix.to(device)
byte_lengths = byte_lengths.to(device)

avg_bytes = byte_lengths.float().mean().item()
print(f'Avg bytes/token    : {avg_bytes:.2f}')
print(f'Byte length dist   : {[(byte_lengths == i).sum().item() for i in range(1, MAX_BYTES + 1)]}')

### 3. Synthetic Data

In [ ]:
# -----------------------------------------------------------------------
# Markov sequences with cluster structure.
# -----------------------------------------------------------------------

N_CLUSTERS         = 16
tokens_per_cluster = VOCAB // N_CLUSTERS
transition         = torch.zeros(VOCAB, VOCAB)

for c in range(N_CLUSTERS):
    s, e = c * tokens_per_cluster, (c + 1) * tokens_per_cluster
    transition[s:e, s:e] = 1.0
    next_c = (c + 1) % N_CLUSTERS
    ns, ne = next_c * tokens_per_cluster, (next_c + 1) * tokens_per_cluster
    transition[s:e, ns:ne] = 0.3
    rand_c = (c + 3) % N_CLUSTERS
    rs, re = rand_c * tokens_per_cluster, (rand_c + 1) * tokens_per_cluster
    transition[s:e, rs:re] = 0.1

transition = transition / transition.sum(dim=-1, keepdim=True)


def generate_sequences(n_seqs, seq_len):
    seqs       = torch.zeros(n_seqs, seq_len + 1, dtype=torch.long)
    seqs[:, 0] = torch.randint(0, VOCAB, (n_seqs,))
    for t in range(seq_len):
        probs          = transition[seqs[:, t]]
        seqs[:, t + 1] = torch.multinomial(probs, 1).squeeze(-1)
    return seqs


train_data = generate_sequences(2000, SEQ_LEN).to(device)
val_data   = generate_sequences(500,  SEQ_LEN).to(device)
print(f'Train : {train_data.shape}  |  Val : {val_data.shape}')

### 4. Shared Transformer Components

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return F.rms_norm(x, (x.size(-1),)) * self.scale


class SimpleAttention(nn.Module):
    '''GQA-style attention with n_heads query heads and n_kv key/value heads.'''
    def __init__(self, dim, n_heads=NUM_HEADS, n_kv=NUM_KV_HEADS):
        super().__init__()
        self.n_heads  = n_heads
        self.n_kv     = n_kv
        self.head_dim = dim // n_heads
        self.qkv      = nn.Linear(dim, dim + 2 * (n_kv * self.head_dim), bias=False)
        self.proj     = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, S, D  = x.shape
        q_size   = self.n_heads * self.head_dim
        kv_size  = self.n_kv   * self.head_dim
        q, k, v  = self.qkv(x).split([q_size, kv_size, kv_size], dim=-1)
        q        = q.reshape(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k        = k.reshape(B, S, self.n_kv,   self.head_dim).transpose(1, 2)
        v        = v.reshape(B, S, self.n_kv,   self.head_dim).transpose(1, 2)
        rep      = self.n_heads // self.n_kv
        k        = k.repeat_interleave(rep, dim=1)
        v        = v.repeat_interleave(rep, dim=1)
        scale    = 1.0 / math.sqrt(self.head_dim)
        attn     = (q @ k.transpose(-2, -1)) * scale
        mask     = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        attn     = F.softmax(attn.masked_fill(mask, float('-inf')), dim=-1)
        return self.proj((attn @ v).transpose(1, 2).reshape(B, S, D))


class SimpleMLP(nn.Module):
    def __init__(self, dim, mult=MLP_MULT):
        super().__init__()
        self.fc   = nn.Linear(dim, dim * mult, bias=False)
        self.proj = nn.Linear(dim * mult, dim, bias=False)
    def forward(self, x):
        return self.proj(F.relu(self.fc(x)).square())


class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = SimpleAttention(dim)
        self.norm2 = RMSNorm(dim)
        self.mlp   = SimpleMLP(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

### 5. Option A -- Sequence Refiner

In [ ]:
class CausalConvRefiner(nn.Module):
    '''
    Causal 1D depthwise-separable conv refiner with n_layers stages.
    Each stage pads causally, applies depthwise then pointwise conv.
    Intermediate stages apply GELU. Zero-init gate blends in the
    refiner residual gradually at the start of training.
    '''
    def __init__(self, dim, kernel_size=3, n_layers=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.dw   = nn.ModuleList([
            nn.Conv1d(dim, dim, kernel_size, padding=0, groups=dim, bias=False)
            for _ in range(n_layers)
        ])
        self.pw   = nn.ModuleList([
            nn.Conv1d(dim, dim, 1, bias=False)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(dim)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, S, D = x.shape
        h       = x.permute(0, 2, 1).contiguous()
        for i, (dw, pw) in enumerate(zip(self.dw, self.pw)):
            h = F.pad(h, (self.kernel_size - 1, 0))
            h = pw(dw(h))
            if i < len(self.dw) - 1:
                h = F.gelu(h)
        h = h.permute(0, 2, 1).contiguous()
        return x + torch.tanh(self.gate) * self.norm(h)

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

### 6. Option B -- ByteCNN Vocabulary Generator

In [ ]:
class ByteCNNEmbedding(nn.Module):
    '''
    Generates the [VOCAB, embed_dim] embedding matrix from byte spellings,
    replacing nn.Embedding(VOCAB, embed_dim) entirely.

    During training: runs CNN on all vocab byte sequences every forward pass.
    During eval: caches the generated matrix (vocab does not change).
    '''
    def __init__(self, vocab_size, embed_dim, byte_matrix, byte_lengths,
                 byte_embed_dim=32, cnn_channels=64, kernel_size=3):
        super().__init__()
        self.vocab_size  = vocab_size
        self.embed_dim   = embed_dim
        self.register_buffer('byte_matrix',  byte_matrix)
        self.register_buffer('byte_lengths', byte_lengths)
        self.byte_emb    = nn.Embedding(257, byte_embed_dim, padding_idx=256)
        self.conv1       = nn.Conv1d(byte_embed_dim, cnn_channels, kernel_size,
                                     padding=kernel_size // 2)
        self.conv2       = nn.Conv1d(cnn_channels,   cnn_channels, kernel_size,
                                     padding=kernel_size // 2)
        self.proj        = nn.Linear(cnn_channels, embed_dim, bias=False)
        self._cached_emb = None

    def _build_matrix(self):
        bm           = self.byte_matrix.clone()
        seq_positions = torch.arange(bm.size(1), device=bm.device).unsqueeze(0)
        pad_mask     = seq_positions >= self.byte_lengths.unsqueeze(1)
        bm.masked_fill_(pad_mask, 256)
        x            = self.byte_emb(bm).transpose(1, 2)
        x            = F.gelu(self.conv1(x))
        x            = F.gelu(self.conv2(x))
        pool_mask    = (~pad_mask).unsqueeze(1).float()
        x            = (x * pool_mask + (1 - pool_mask) * (-1e9)).max(dim=-1).values
        return self.proj(x)

    def forward(self, input_ids):
        if self.training:
            emb_matrix       = self._build_matrix()
            self._cached_emb = None
        else:
            if self._cached_emb is None:
                self._cached_emb = self._build_matrix().detach()
            emb_matrix = self._cached_emb
        return F.embedding(input_ids, emb_matrix)

    def get_weight(self):
        '''For tied output head -- returns the generated matrix.'''
        if self.training:
            return self._build_matrix()
        if self._cached_emb is None:
            self._cached_emb = self._build_matrix().detach()
        return self._cached_emb

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

### 7. Model Variants

In [ ]:
class BaselineModel(nn.Module):
    '''Standard: nn.Embedding + embed_proj + transformer + embed_proj_rev + tied head.'''
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.tok_emb        = nn.Embedding(VOCAB, embed_dim)
        self.embed_proj     = nn.Linear(embed_dim, DIM, bias=False)
        self.embed_proj_rev = nn.Linear(DIM, embed_dim, bias=False)
        self.blocks         = nn.ModuleList([TransformerBlock(DIM) for _ in range(N_LAYERS)])
        self.norm           = RMSNorm(DIM)

    def _encode(self, idx):
        x = self.embed_proj(self.tok_emb(idx))
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

    def forward(self, idx):
        proj = self.embed_proj_rev(self._encode(idx))
        return F.linear(proj, self.tok_emb.weight)

    def embedding_params(self):
        return (self.tok_emb.weight.numel() +
                self.embed_proj.weight.numel() +
                self.embed_proj_rev.weight.numel())

    def embedding_bytes_fp16(self):
        return self.embedding_params() * 2


class RefinerModel(BaselineModel):
    '''Baseline + CausalConvRefiner applied before the output projection.'''
    def __init__(self, kernel_size=3, n_layers=1):
        super().__init__()
        self.refiner = CausalConvRefiner(DIM, kernel_size=kernel_size, n_layers=n_layers)

    def forward(self, idx):
        proj = self.embed_proj_rev(self.refiner(self._encode(idx)))
        return F.linear(proj, self.tok_emb.weight)

    def extra_params(self):
        return self.refiner.param_count()


class ByteCNNModel(nn.Module):
    '''
    ByteCNN generates embeddings from byte spellings via a small CNN.
    Tied output head. embed_dim can be set freely since the output is
    not a vocabulary lookup -- useful for shrinking embedding storage.
    '''
    def __init__(self, embed_dim=EMBED_DIM, byte_embed_dim=32, cnn_channels=64):
        super().__init__()
        self.tok_emb        = ByteCNNEmbedding(VOCAB, embed_dim, byte_matrix, byte_lengths,
                                               byte_embed_dim=byte_embed_dim,
                                               cnn_channels=cnn_channels)
        self.embed_proj     = nn.Linear(embed_dim, DIM, bias=False)
        self.embed_proj_rev = nn.Linear(DIM, embed_dim, bias=False)
        self.blocks         = nn.ModuleList([TransformerBlock(DIM) for _ in range(N_LAYERS)])
        self.norm           = RMSNorm(DIM)

    def forward(self, idx):
        x    = self.embed_proj(self.tok_emb(idx))
        for block in self.blocks:
            x = block(x)
        proj = self.embed_proj_rev(self.norm(x))
        return F.linear(proj, self.tok_emb.get_weight())

    def embedding_params(self):
        return (self.tok_emb.param_count() +
                self.embed_proj.weight.numel() +
                self.embed_proj_rev.weight.numel())

    def embedding_bytes_fp16(self):
        return self.embedding_params() * 2

### 8. Training and Evaluation

In [ ]:
def train_and_eval(model, name, n_steps=N_STEPS):
    model = model.to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    for _ in range(5):
        batch  = train_data[torch.randint(0, len(train_data), (BATCH,), device=device)]
        logits = model(batch[:, :-1])
        loss   = F.cross_entropy(logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1))
        loss.backward()
        opt.zero_grad()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    for _ in range(1, n_steps + 1):
        model.train()
        batch  = train_data[torch.randint(0, len(train_data), (BATCH,), device=device)]
        logits = model(batch[:, :-1])
        loss   = F.cross_entropy(logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

    elapsed = time.perf_counter() - t0
    ms_step = elapsed / n_steps * 1000

    model.eval()
    with torch.no_grad():
        val_logits = model(val_data[:, :-1])
        val_loss   = F.cross_entropy(val_logits.reshape(-1, VOCAB),
                                     val_data[:, 1:].reshape(-1)).item()

    return {'name': name, 'val_loss': val_loss, 'ms_step': ms_step}

### 9. Run Experiments

In [ ]:
configs = [
    ('Baseline (emb=256)',        BaselineModel()),
    ('Refiner k=3 (1 layer)',     RefinerModel(kernel_size=3, n_layers=1)),
    ('Refiner k=5 (1 layer)',     RefinerModel(kernel_size=5, n_layers=1)),
    ('Refiner k=3 (2 layer)',     RefinerModel(kernel_size=3, n_layers=2)),
    ('Refiner k=5 (2 layer)',     RefinerModel(kernel_size=5, n_layers=2)),
    ('ByteCNN (emb=256, ch=64)',  ByteCNNModel(embed_dim=256, byte_embed_dim=32, cnn_channels=64)),
    ('ByteCNN (emb=256, ch=128)', ByteCNNModel(embed_dim=256, byte_embed_dim=32, cnn_channels=128)),
    ('ByteCNN (emb=128)',         ByteCNNModel(embed_dim=128, byte_embed_dim=32, cnn_channels=64)),
    ('ByteCNN (emb=64)',          ByteCNNModel(embed_dim=64,  byte_embed_dim=32, cnn_channels=64)),
]

# -----------------------------------------------------------------------
# Storage analysis before training.
# -----------------------------------------------------------------------

print(f'\n{"Model":<30} {"EmbParams":>10} {"EmbMB fp16":>10} {"ExtraParams":>10} {"TotalParams":>10} {"SavedMB":>8}')
print('-' * 80)

baseline_emb_bytes = None
for name, model in configs:
    total  = sum(p.numel() for p in model.parameters())
    emb_p  = model.embedding_params() if hasattr(model, 'embedding_params') else 0
    emb_mb = emb_p * 2 / 1e6
    extra  = model.extra_params()      if hasattr(model, 'extra_params')    else 0
    if baseline_emb_bytes is None:
        baseline_emb_bytes = emb_mb
        saved = 0.0
    else:
        saved = baseline_emb_bytes - emb_mb
    print(f'{name:<30} {emb_p:>10,} {emb_mb:>9.2f}M {extra:>10,} {total:>10,} {saved:>+7.2f}M')

# -----------------------------------------------------------------------
# Training comparison.
# -----------------------------------------------------------------------

print(f'\n{"Model":<30} {"ValLoss":>8} {"ms/step":>8} {"Notes":>30}')
print('-' * 80)

baseline_loss = None
for name, model in configs:
    r = train_and_eval(model, name)
    if baseline_loss is None:
        baseline_loss = r['val_loss']
    delta = r['val_loss'] - baseline_loss
    sign  = '+' if delta >= 0 else ''
    notes = f'{sign}{delta:.4f} vs baseline'
    print(f'{name:<30} {r["val_loss"]:>8.4f} {r["ms_step"]:>7.1f}ms {notes:>30}')

print(f'Standard embedding: {VOCAB}*{EMBED_DIM} = {VOCAB * EMBED_DIM:,} params = {VOCAB * EMBED_DIM * 2 / 1e6:.2f}MB fp16')